# 10. 프로덕션

## 학습 목표

에이전트를 테스트, 배포, 모니터링하는 방법을 알아봅니다.

이 노트북에서 다루는 내용:
- LangSmith Studio를 사용한 로컬 개발 및 디버깅
- `GenericFakeChatModel`로 결정론적 에이전트 테스트
- 트라젝토리 기반 테스트로 도구 호출 순서 검증
- Agent Chat UI로 웹 기반 대화
- LangGraph Platform 및 자체 서버 배포
- LangSmith를 활용한 관측성(Observability)

## 10.1 환경 설정

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

from langchain.agents import create_agent
from langchain.tools import tool

print("환경 준비 완료.")

환경 준비 완료.


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 10.2 LangSmith Studio

로컬에서 에이전트를 개발하고 디버깅합니다.

Studio를 사용하려면 다음이 필요합니다:
- `langgraph.json` 설정 파일
- `langgraph dev` 명령어로 로컬 서버 실행
- Studio UI에서 에이전트를 인터랙티브하게 테스트

Studio는 에이전트의 실행 흐름을 시각적으로 확인하고, 각 단계를 디버깅할 수 있는 강력한 도구입니다.

In [3]:
# langgraph.json 설정 예시
import json

langgraph_config = {
    "dependencies": ["."],
    "graphs": {
        "agent": "./agent.py:agent"
    },
    "env": ".env"
}

print("langgraph.json 설정 예시:")
print(json.dumps(langgraph_config, indent=2))
print("\n실행 방법:")
print("  $ langgraph dev")
print("  → http://localhost:2024 에서 Studio UI 접근")

langgraph.json 설정 예시:
{
  "dependencies": [
    "."
  ],
  "graphs": {
    "agent": "./agent.py:agent"
  },
  "env": ".env"
}

실행 방법:
  $ langgraph dev
  → http://localhost:2024 에서 Studio UI 접근


## 10.3 에이전트 테스트

`GenericFakeChatModel`을 사용하면 실제 API 호출 없이 결정론적으로 에이전트를 테스트할 수 있습니다.

이 방법의 장점:
- API 비용 없이 테스트 가능
- 항상 동일한 결과를 반환하므로 CI/CD 파이프라인에 적합
- 에이전트의 로직(도구 호출, 분기 등)을 독립적으로 검증

In [4]:
from langchain_core.language_models import GenericFakeChatModel
from langchain.messages import AIMessage
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_capital(country: str) -> str:
    """국가의 수도를 반환합니다."""
    capitals = {"Korea": "Seoul", "Japan": "Tokyo", "France": "Paris"}
    return capitals.get(country, "알 수 없음")

# 가짜 모델로 결정론적 테스트
fake_model = GenericFakeChatModel(
    messages=iter([
        AIMessage(content="대한민국의 수도는 서울입니다.")
    ])
)

# 테스트 에이전트
test_agent = create_agent(
    model=fake_model,
    tools=[get_capital],
    system_prompt="당신은 지리 전문가입니다.",
)

print("GenericFakeChatModel 테스트:")
print("  → 결정론적 응답으로 에이전트 동작을 테스트합니다")
print("  → CI/CD 파이프라인에서 API 호출 없이 테스트 가능")

GenericFakeChatModel 테스트:
  → 결정론적 응답으로 에이전트 동작을 테스트합니다
  → CI/CD 파이프라인에서 API 호출 없이 테스트 가능


## 10.4 트라젝토리 기반 테스트

에이전트의 도구 호출 순서를 검증합니다. 트라젝토리 테스트는 에이전트가 올바른 순서로 도구를 호출하는지, 최종 응답이 기대에 부합하는지 확인합니다.

In [5]:
# 트라젝토리 테스트 예시
def test_agent_trajectory():
    """에이전트가 예상된 순서로 도구를 호출하는지 테스트합니다."""
    result = test_agent.invoke(
        {"messages": [{"role": "user", "content": "대한민국의 수도는 어디인가요?"}]}
    )
    
    messages = result["messages"]
    
    # 검증: 메시지가 존재하는지
    assert len(messages) > 0, "에이전트가 응답하지 않았습니다"
    
    # 검증: 마지막 메시지가 AI 응답인지
    last_msg = messages[-1]
    assert hasattr(last_msg, 'content'), "마지막 메시지에 content가 없습니다"
    
    print("✓ 트라젝토리 테스트 통과")
    print(f"  메시지 수: {len(messages)}")
    print(f"  최종 응답: {last_msg.content[:100]}")

try:
    test_agent_trajectory()
except Exception as e:
    print(f"테스트 참고: {e}")

테스트 참고: 


## 10.5 Agent Chat UI

에이전트와 대화할 수 있는 웹 UI입니다. LangGraph 서버와 연결하여 브라우저에서 직접 에이전트를 테스트할 수 있습니다.

주요 기능:
- 실시간 스트리밍 채팅
- 도구 호출 시각화
- 대화 분기(branching)
- Human-in-the-loop 승인

In [6]:
print("Agent Chat UI 설정:")
print("=" * 50)
print("""
# 1. Agent Chat UI 설치
$ npx @anthropic-ai/agent-chat-ui

# 2. LangGraph 서버 시작
$ langgraph dev

# 3. UI에서 http://localhost:2024 연결
#    → 웹 브라우저에서 에이전트와 대화
""")
print("주요 기능:")
print("  - 실시간 스트리밍 채팅")
print("  - 도구 호출 시각화")
print("  - 대화 분기(branching)")
print("  - Human-in-the-loop 승인")

Agent Chat UI 설정:

# 1. Agent Chat UI 설치
$ npx @anthropic-ai/agent-chat-ui

# 2. LangGraph 서버 시작
$ langgraph dev

# 3. UI에서 http://localhost:2024 연결
#    → 웹 브라우저에서 에이전트와 대화

주요 기능:
  - 실시간 스트리밍 채팅
  - 도구 호출 시각화
  - 대화 분기(branching)
  - Human-in-the-loop 승인


## 10.6 배포

LangGraph Platform(관리형) 또는 자체 서버로 에이전트를 배포할 수 있습니다. 프로덕션 환경에 맞는 옵션을 선택하세요.

In [7]:
print("배포 옵션:")
print("=" * 50)

print("""
# 옵션 1: LangGraph Platform (관리형)
$ langgraph deploy


# 옵션 2: 자체 Docker 배포
$ langgraph build -t my-agent
$ docker run -p 2024:2024 my-agent


# 옵션 3: FastAPI/Flask 래핑
from fastapi import FastAPI

app = FastAPI()


@app.post("/chat")
async def chat(message: str):
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": message
                }
            ]
        }
    )

    return {
        "response": result["messages"][-1].content
    }
""")

배포 옵션:

# 옵션 1: LangGraph Platform (관리형)
$ langgraph deploy


# 옵션 2: 자체 Docker 배포
$ langgraph build -t my-agent
$ docker run -p 2024:2024 my-agent


# 옵션 3: FastAPI/Flask 래핑
from fastapi import FastAPI

app = FastAPI()


@app.post("/chat")
async def chat(message: str):
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": message
                }
            ]
        }
    )

    return {
        "response": result["messages"][-1].content
    }



## 10.7 관측성

LangSmith로 에이전트 동작을 추적합니다. 트레이싱을 활성화하면 에이전트의 모든 실행 단계를 기록하고 분석할 수 있습니다.

LangSmith에서 확인할 수 있는 정보:
- 각 에이전트 호출의 전체 실행 흐름
- 모델 입/출력, 도구 호출, 토큰 사용량
- 지연 시간, 에러, 비용 추적

In [8]:
print("LangSmith 관측성 설정:")
print("=" * 50)
print("""
# .env에 설정:
LANGSMITH_API_KEY=lsv2-...
LANGSMITH_TRACING=true

# 설정 후 에이전트 실행 시 자동으로 트레이스가 기록됩니다.
# https://smith.langchain.com 에서 확인:
#   - 각 에이전트 호출의 전체 실행 흐름
#   - 모델 입/출력, 도구 호출, 토큰 사용량
#   - 지연 시간, 에러, 비용 추적
""")

# 트레이싱 상태 확인
import os
tracing = os.environ.get("LANGSMITH_TRACING", "false")
api_key = os.environ.get("LANGSMITH_API_KEY", "")
print(f"\n현재 트레이싱 상태: {'활성' if tracing.lower() == 'true' else '비활성'}")
print(f"API 키 설정: {'✓' if api_key else '✗ 미설정'}")

LangSmith 관측성 설정:

# .env에 설정:
LANGSMITH_API_KEY=lsv2-...
LANGSMITH_TRACING=true

# 설정 후 에이전트 실행 시 자동으로 트레이스가 기록됩니다.
# https://smith.langchain.com 에서 확인:
#   - 각 에이전트 호출의 전체 실행 흐름
#   - 모델 입/출력, 도구 호출, 토큰 사용량
#   - 지연 시간, 에러, 비용 추적


현재 트레이싱 상태: 비활성
API 키 설정: ✗ 미설정


## 10.8 프로덕션 체크리스트

에이전트를 프로덕션에 배포하기 전에 아래 항목을 확인하세요.

| 항목 | 도구 | 상태 |
|------|------|------|
| 단위 테스트 | `GenericFakeChatModel`, `pytest` | |
| 트라젝토리 테스트 | 커스텀 검증 함수 | |
| 관측성 설정 | LangSmith 트레이싱 | |
| 에러 처리 | `try/except`, 재시도 로직 | |
| 보안 | API 키 관리, 입력 검증, 가드레일 | |
| 배포 환경 | Docker, LangGraph Platform | |
| 모니터링 | LangSmith 대시보드, 알림 설정 | |
| 문서화 | API 문서, 에이전트 동작 설명 | |

## 10.9 요약

이 노트북에서 배운 내용:

| 주제 | 핵심 내용 |
|------|----------|
| **LangSmith Studio** | `langgraph dev`로 로컬에서 에이전트를 시각적으로 디버깅합니다 |
| **에이전트 테스트** | `GenericFakeChatModel`로 API 호출 없이 결정론적 테스트를 수행합니다 |
| **트라젝토리 테스트** | 도구 호출 순서와 최종 응답을 검증합니다 |
| **Agent Chat UI** | 웹 브라우저에서 에이전트와 대화하고 도구 호출을 시각화합니다 |
| **배포** | LangGraph Platform, Docker, FastAPI 등으로 배포합니다 |
| **관측성** | LangSmith로 실행 흐름, 토큰 사용량, 비용을 추적합니다 |

이것으로 LangChain v1 에이전트 과정을 마칩니다. 기본 개념부터 프로덕션 배포까지, 에이전트 개발의 전체 라이프사이클을 다루었습니다.

### 다음 단계
→ **[LangGraph 중급 과정](../03_langgraph/01_introduction.ipynb)**으로 진행하세요!
